The Task: 

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a **Gradio UI**, **streaming**, use of the **system prompt** to add expertise, and the ability to **switch between models**. Bonus points if you can demonstrate use of a tool!

**Keys:** You only need `OPENROUTER_API_KEY` in your `.env`. OpenRouter lets you call Gemini, GPT-4, Claude, etc. with that single key — no extra keys needed.

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

# 1. Load credentials — only OPENROUTER_API_KEY needed (same as Week 1)
load_dotenv()

# 2. OpenRouter client (same setup as Week 1)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# 3. System prompt — adds expertise (same role as Week 1)
system_prompt = "You are a helpful technical tutor who answers questions about software engineering, Python, and data science."
user_prompt_template = "Please provide a detailed explanation for this question: {}"

# 4. Models you can switch between (all via OpenRouter, one key)
MODELS = [
    "google/gemini-2.0-flash-001",
    "openai/gpt-4o",
    "anthropic/claude-3.5-sonnet",
]

# 5. Streaming response: system prompt + selected model (like Week 1, but with model choice)
def get_response(question, selected_model):
    if not (question or "").strip():
        yield "Please enter a question."
        return
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_template.format(question.strip())}
    ]
    try:
        stream = client.chat.completions.create(
            model=selected_model,
            messages=messages,
            stream=True,
            extra_headers={
                "HTTP-Referer": "http://localhost:3000",
                "X-Title": "Technical Tutor Tool",
            }
        )
        yield "Tutor is generating your explanation...\n\n"
        response = ""
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                response += content
                yield response
    except Exception as e:
        yield f"❌ Error: {e}"

# 6. Gradio UI: question input, model selector, streaming answer
with gr.Blocks(title="Technical Q&A") as demo:
    gr.Markdown("## Technical question/answerer — Gradio UI, streaming, system prompt, multi-model")
    question_input = gr.Textbox(
        label="Ask a technical question",
        placeholder="e.g. Explain the difference between a list comprehension and a generator expression in Python.",
        lines=3
    )
    with gr.Row():
        model_selector = gr.Dropdown(choices=MODELS, value=MODELS[0], label="Select model")
        submit_btn = gr.Button("Get answer")
    answer_output = gr.Markdown(label="Answer")

    submit_btn.click(
        fn=get_response,
        inputs=[question_input, model_selector],
        outputs=answer_output
    )

demo.launch(inline=True)

C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
